# YOLOv8 Bone Fracture Detection - Model Training

This notebook trains a YOLOv8 model on the cleaned bone fracture detection dataset.

In [8]:
# Install ultralytics (YOLOv8)
%pip install ultralytics
import torch
torch.cuda.empty_cache()

In [9]:
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import yaml
import torch

# Check what devices are available
print("Device Check:")
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  MPS (Apple GPU) available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")
print(f"  CPU: Always available")

# Determine best device
if torch.cuda.is_available():
    device = 0
    device_name = "CUDA GPU"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'  # Apple Silicon GPU
    device_name = "Apple MPS (GPU)"
else:
    device = 'cpu'
    device_name = "CPU"

print(f"\nUsing device: {device_name} ({device})")

Device Check:
  CUDA available: True
  MPS (Apple GPU) available: False
  CPU: Always available

Using device: CUDA GPU (0)


In [ ]:
import kagglehub
import os

# download dataset (or locate cached version)
top_level_path = kagglehub.dataset_download(
    "pkdarabi/bone-fracture-detection-computer-vision-project"
)

# dataset folder
data_path = os.path.join(top_level_path, "BoneFractureYolo8")

# yaml file
data_yaml_path = os.path.join(data_path, "data.yaml")

print("Using dataset:", data_yaml_path)

Using Colab cache for faster access to the 'bone-fracture-detection-computer-vision-project' dataset.
Using dataset: /kaggle/input/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/data.yaml


In [ ]:
model = YOLO('yolov8m.pt')
print(f"Model initialized: {model.model_name}")

Model initialized: yolov8m.pt


In [ ]:
# Train the model
results = model.train(
    data=data_yaml_path,
    epochs=10,              # Number of training epochs
    imgsz=768,              # Image size
    batch=16 if device != 'cpu' else 4,  # Further reduced batch size for GPU
    name='bone_fracture_yolov8',  # Project name
    project='runs/detect',  # Project directory
    patience=20,            # Early stopping patience
    save=True,              # Save checkpoints
    plots=True,             # Generate training plots
    val=True,               # Validate during training
    device=device,          # Uses device detected in Cell 2 (MPS/GPU/CPU)
    lr0=0.0005,             # learning rate
    workers=8 if device != 'cpu' else 4  # More workers for GPU
)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bone_fracture_yolov82, nbs=64, nms=False, opset=None, o

## Training Results

After training, the model will be saved in `runs/detect/bone_fracture_yolov8/weights/best.pt`

In [ ]:
print("Starting evaluation...")

best_model_path = results.save_dir / "weights" / "best.pt"
model = YOLO(str(best_model_path))

metrics = model.val(data=data_yaml_path, verbose=False)

print("Validation finished!")


metrics = model.val(data=data_yaml_path)
print("\nValidation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")



Starting evaluation...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,843,813 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 42.5±26.6 MB/s, size: 10.4 KB)
val: Scanning /kaggle/input/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/valid/labels... 348 images, 175 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 348/348 488.1it/s 0.7s
WARNING ⚠️ val: Cache directory /kaggle/input/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 1.9it/s 11.4s
                   all        348        204      0.282      0.122     0.0799     0.0293
Speed: 1.9ms preprocess, 25.8ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/runs/detect/val2
Validation finished!

===== Model Evaluation 